Today's topics:
* a center number and a spread number
* mean, median, variance, standard deviation: by hand, then with NumPy

# What is the hardness of this steel?

20 Rockwell C readings, one piece of heat-treated steel, one tester, one
afternoon. Run the cell below; you don't need to read its code.

<img src="https://upload.wikimedia.org/wikipedia/commons/b/bf/Rockwell_hardness_tester_001.jpg" alt="Rockwell hardness tester" height=200/>

(Three-quarter-ten, [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Rockwell_hardness_tester_001.jpg), [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/))

## Dataset: Repeated Rockwell Hardness (HRC) Readings

**Provenance:** these are synthetic teaching values, designed to resemble repeated
measurements from heat-treated steel. They are not records from a published
experiment.

`hardness` represents 20 Rockwell-C hardness readings at different points on the
*same* specimen. In principle a homogeneous specimen should give the same number
every time, but indentation tests have measurement noise (probe placement, local
microstructure, surface finish), so the simulated readings have a realistic spread.
One reading (index 19) is noticeably higher than the rest -- worth keeping an eye on
as we compare mean and median.

`furnace_A` and `furnace_B` represent 20 readings each from two simulated
heat-treatment batches. Both batches target the same hardness, while batch B was
constructed with substantially more variability. Use them to compare two batches
with (nearly) the same mean but very different spread.

In [1]:
import numpy as np

hardness = np.array([58.2, 57.9, 58.6, 57.5, 58.9, 58.1, 57.8, 58.4, 58.0, 57.6,
                      58.3, 58.7, 57.4, 58.2, 58.5, 57.9, 58.1, 58.8, 57.7, 63.9])

furnace_A = np.array([58.6, 58.5, 58.2, 58.5, 57.9, 57.4, 58.9, 58.4, 57.3, 57.7,
                       57.9, 58.3, 58.8, 58.3, 58.0, 58.4, 58.1, 58.3, 57.9, 57.7])
furnace_B = np.array([58.3, 57.4, 59.5, 56.2, 60.4, 58.0, 57.1, 58.9, 57.7, 56.5,
                       58.6, 59.8, 55.9, 58.3, 59.2, 57.4, 58.0, 60.1, 56.8, 59.5])

print(f'{len(hardness)} hardness readings, units HRC')
print(hardness)

20 hardness readings, units HRC
[58.2 57.9 58.6 57.5 58.9 58.1 57.8 58.4 58.  57.6 58.3 58.7 57.4 58.2
 58.5 57.9 58.1 58.8 57.7 63.9]


`[board]` indenter, specimen, depth. Bigger number, harder steel.

So what is the hardness? "About 58." The data sheet has one box. 58.2? 58.1?
58.6? All in the list. So is 63.9.

`[board]` all twenty on one axis, stacked where they repeat. A pile in the high
57s and 58s, one reading well clear.

Twenty numbers describe the pile exactly. Nobody carries twenty numbers. So we
squeeze: where is the pile, how wide, does 63.9 belong. One number per answer.
Each number represents one property and nothing else. Today: a center number
and a spread
number.

# Warm-up

Five numbers we can check in our heads.

In [2]:
import numpy as np

warmup = np.array([2, 4, 4, 6, 9])
print(f'mean:   {np.mean(warmup)}')
print(f'median: {np.median(warmup)}')

mean:   5.0
median: 4.0


Sum 25, five values, mean 5. Sorted middle value 4. Already different.

# Mean

First question: where is the pile.

$$\bar{x} = \frac{1}{N}\sum_{i=1}^N x_i$$

Add everything up, divide by how many. Let's build it before we call it.

In [3]:
# LIVE: mean by hand

In [4]:
print(f'mean (np.mean): {np.mean(hardness):.4f}')

mean (np.mean): 58.4250


`[board]` readings as weights on a ruler, fulcrum at the mean.

*Is the mean sensitive to one unusual value?* Copy the array, then push the last
reading up. `.copy()` so the original stays unchanged; `array[i] = value` to write into
position `i`.

In [5]:
# LIVE: copy, push the last reading up, compare the means

One value out of twenty, and the mean moved by three quarters of a point.

# Median

Sort, then walk to the middle. 20 readings is even, so average positions 9
and 10.

In [6]:
# LIVE: sort, then average the two middle entries

In [7]:
print(f'median (np.median): {np.median(hardness):.4f}')

median (np.median): 58.1500


Same experiment, both statistics:

In [8]:
# LIVE: original against inflated, mean and median side by side

The mean jumped; the median did not move. **Robustness.** The median depends
only on rank order, and the largest reading was already the largest.

`[board]` two rulers, original and inflated, mean and median ticks on each.

### [Check your understanding]

We pushed one reading up. Push one down.

1. Make a fresh copy of `hardness` with `.copy()`
2. Set index 0 to `40.0`
3. Print mean and median of the original and your copy
4. Which moved more? Is that the effect one low outlier should have?

*Optional challenge*

5. How far down does that one reading have to go to move the median a full
   point?

# Variance

Two answers to where the pile is, none yet to how wide. *How far is each
reading from the mean?* Subtract, then try averaging the result.

In [9]:
# LIVE: deviations from the mean, then try averaging them

Zero. Every time, for any data. That is what balance point means: the positive
and negative deviations cancel exactly.

`[board]` arrows from the mean to each reading; left and right cancel.

Squaring removes the signs:

$$\sigma^2 = \frac{1}{N}\sum_{i=1}^N (x_i - \bar{x})^2$$

In [10]:
# LIVE: square first, then average

`np.var` divides by $N$. Your stats textbook may divide by $N - 1$; that's
`ddof=1`. Population versus sample: describing these 20 readings, or estimating
the batch they came from. Your call, before you call the function.

In [11]:
print(f'variance (np.var):         {np.var(hardness):.4f}')
print(f'variance (np.var, ddof=1): {np.var(hardness, ddof=1):.4f}')

variance (np.var):         1.7509
variance (np.var, ddof=1): 1.8430


# Standard deviation

Variance is in $\mathrm{HRC}^2$. Taking the square root returns the units to HRC.

$$\sigma = \sqrt{\sigma^2}$$

In [12]:
# LIVE: square root brings the units back

In [13]:
print(f'standard deviation (np.std): {np.std(hardness):.4f}')

standard deviation (np.std): 1.3232


### [Check your understanding]

Variance and standard deviation of `warmup`, `[2, 4, 4, 6, 9]`, by hand. No
`np.var`, no `np.std` until step 5.

1. Mean
2. Deviations
3. Square, then average. That's the variance
4. `np.sqrt`. That's the standard deviation
5. Check against `np.var` and `np.std`

*Optional challenge*

6. Repeat with `ddof=1`. How big is the gap at $N = 5$?

# Two furnaces

Same steel, same heat treatment, same target. Twenty parts each, loaded as
`furnace_A` and `furnace_B`.

In [14]:
print(f'furnace A -- mean: {np.mean(furnace_A):.2f}, std: {np.std(furnace_A):.4f}')
print(f'furnace B -- mean: {np.mean(furnace_B):.2f}, std: {np.std(furnace_B):.4f}')

furnace A -- mean: 58.16, std: 0.4201
furnace B -- mean: 58.18, std: 1.2848


Means within 0.02 HRC. Standard deviations differ by a factor of three.

Spec: 58.2 plus or minus 1 HRC. `np.min` and `np.max` from L02:

In [15]:
print(f'furnace A -- min: {np.min(furnace_A)}, max: {np.max(furnace_A)}')
print(f'furnace B -- min: {np.min(furnace_B)}, max: {np.max(furnace_B)}')

furnace A -- min: 57.3, max: 58.9
furnace B -- min: 55.9, max: 60.4


`[board]` band 57.2 to 59.2. A's twenty inside. B's ten outside.

Judged by the mean alone, the furnaces look the same. Judged by the standard deviation, one furnace's output
ships and half of the other's is scrapped.

And the third question, does 63.9 belong: it sits more than four standard
deviations out. Deciding that it's unusual is a statistics question. Deciding *why* (edge, prior
indent, scale, or a harder pocket of microstructure) is a materials question.

# One number is never enough

The center is where the material sits. The spread is how much we can count on
that. Report one without the other and two furnaces look identical on paper.

## Careful testing comes first

Every number today describes scatter inside the sample we have, and nothing
about whether that sample resembles the parts in service.

1954, two de Havilland Comet 1 airliners broke apart in flight from metal
fatigue. The test fuselage had lasted far longer, but it had been proof-tested
first at twice the working pressure. Local plastic flow around the windows,
longer life. Twenty such fuselages would have made a tight pile in the wrong
place. The numbers summarize the test that was run. Whether the right test was
run is a materials question, and it comes first.
([NASA, *Exploring in Aeronautics*, NASA-EP-89](https://ntrs.nasa.gov/citations/19750064998))

## Further reading

- [VanderPlas: Aggregations (min, max, and everything in between)](https://jakevdp.github.io/PythonDataScienceHandbook/02.04-computation-on-arrays-aggregates.html)
- [McKinney: NumPy basics](https://wesmckinney.com/book/numpy-basics) (descriptive statistics section)